### `await`

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۱. <strong>ساده‌سازی اتصال ادامه‌ها با await</strong></h3>
<ul><li><p>کلیدواژه <code>await</code> در سی‌شارپ برای ساده‌سازی اتصال ادامه‌ها (Continuations) به عملیات‌های ناهمزمان استفاده می‌شود.</p></li><li><p>وقتی از <code>await</code> استفاده می‌کنید، کامپایلر سی‌شارپ کد شما را به شکلی تبدیل می‌کند که شبیه به استفاده دستی از <code>GetAwaiter()</code> و <code>OnCompleted()</code> باشد.</p></li></ul>
<h3>۲. <strong>تبدیل کد با await به کد معادل</strong></h3>
</div>

In [ ]:
var result = await expression;
statement(s);

In [ ]:
//کامپایل می شود به کد زیر
var awaiter = expression.GetAwaiter();
awaiter.OnCompleted(() =>
{
    var result = awaiter.GetResult();
    statement(s);
});

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3> <strong>متدهای ناهمزمان (Asynchronous Functions)</strong></h3>
<ul><li><p>متدهایی که با <strong>async</strong> علامت‌گذاری می‌شوند، <strong>متدهای ناهمزمان</strong> نامیده می‌شوند.</p></li><li><p>این متدها می‌توانند یکی از انواع بازگشتی زیر را داشته باشند:</p><ul><li><p><strong>void</strong>: برای متدهایی که نیازی به بازگشت مقدار ندارند.</p></li><li><p><strong>Task</strong>: برای متدهایی که عملیات ناهمزمان انجام می‌دهند اما مقداری باز نمی‌گردانند.</p></li><li><p><strong>Task<span class="ds-markdown-html">&lt;TResult&gt;</span></strong>: برای متدهایی که عملیات ناهمزمان انجام می‌دهند و یک مقدار بازمی‌گردانند.</p></li></ul></li></ul>
<h3><strong>رفتار اجرایی متدهای ناهمزمان</strong></h3>
<ul><li><p>وقتی اجرا به یک عبارت <strong>await</strong> می‌رسد:</p><ol><li><p>اجرا به caller بازمی‌گردد (مانند <strong>yield return</strong> در iteratorها).</p></li><li><p>یک ادامه (Continuation) به تسک مورد انتظار (Awaited Task) متصل می‌شود.</p></li><li><p>پس از تکمیل تسک، اجرا به متد بازمی‌گردد و از همان نقطه ادامه می‌یابد.</p></li><li><p>اگر تسک با خطا مواجه شود، استثنا دوباره پرتاب می‌شود. در غیر این صورت، نتیجه تسک به متغیر انتساب داده می‌شود.</p></li></ol></li></ul>
<h3> <strong>استفاده از await با Taskهای غیرجنریک</strong></h3>
<ul><li><p><strong>await</strong> می‌تواند روی <strong>Task</strong>های غیرجنریک (بدون مقدار بازگشتی) نیز استفاده شود. در این حالت، عبارت <strong>await</strong> یک عبارت <strong>void</strong> تولید می‌کند:</p></li></ul>

<h3><code>async void</code></h3>
<ul><li><p><strong><code>async void</code></strong> عمدتاً در <strong>رویدادها (Event Handlers)</strong> استفاده می‌شود، جایی که امضای متد باید <strong>void</strong> باشد.</p></li><li><p>در سایر موارد، بهتر است از <strong><code>async Task</code></strong> استفاده کنید تا از مشکلات مربوط به استثناها و مدیریت جریان برنامه جلوگیری شود.</p></li><li><p>اگر مجبور به استفاده از <strong><code>async void</code></strong> هستید، مطمئن شوید که استثناها را به درستی مدیریت می‌کنید تا از crash برنامه جلوگیری شود.</p></li></ul>

</div>

#### Capturing local state

In [ ]:
async void DisplayPrimeCounts()
{
    for (int i = 0; i < 10; i++)
        Console.WriteLine(await GetPrimesCountAsync(i * 1000000 + 2, 1000000));
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<ul><li><p>یکی از ویژگی‌های قدرتمند <strong>await</strong> این است که وضعیت متغیرهای محلی و شمارنده‌های حلقه را حفظ می‌کند.</p></li><li><p>وقتی اجرا به <strong>await</strong> می‌رسد، اجرا به caller بازمی‌گردد، اما وضعیت متغیرهای محلی (مانند <code>i</code> در مثال بالا) ذخیره می‌شود.</p></li><li><p>پس از تکمیل تسک، اجرا از همان نقطه ادامه می‌یابد و مقادیر متغیرهای محلی بازیابی می‌شوند.</p></li></ul>
</div>

### **Asynchronous call graph execution**

In [3]:
async Task Go()
{
    var task = PrintAnswerToLife();
    await task; 
    Console.WriteLine("Done");
}

async Task PrintAnswerToLife()
{
    Console.WriteLine("start PrintAnswerToLife");
    var task = GetAnswerToLife();
    int answer = await task; 
    Console.WriteLine(answer);
}

async Task<int> GetAnswerToLife()
{
    Console.WriteLine("start GetAnswerToLife");
    var task = Task.Delay(1000);
    await task; 
    int answer = 21 * 2; 
    return answer;
}

await Go();

start PrintAnswerToLife
start GetAnswerToLife
42
Done


<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۲. <strong>جریان اجرا:</strong></h3>
<h4>الف) <strong>فاز همزمان (Synchronous Phase):</strong></h4>
<ol start="1"><li><p><strong>فراخوانی <code>Go</code>:</strong></p><ul><li><p><code>Go</code> فراخوانی می‌شود.</p></li><li><p><code>PrintAnswerToLife</code> فراخوانی می‌شود و یک <code>Task</code> برمی‌گرداند.</p></li><li><p><code>await task</code> در <code>Go</code> باعث می‌شود اجرای <code>Go</code> متوقف شود و کنترل به caller بازگردد.</p></li></ul></li><li><p><strong>فراخوانی <code>PrintAnswerToLife</code>:</strong></p><ul><li><p><code>PrintAnswerToLife</code> فراخوانی می‌شود.</p></li><li><p><code>GetAnswerToLife</code> فراخوانی می‌شود و یک <code>Task&lt;int&gt;</code> برمی‌گرداند.</p></li><li><p><code>await task</code> در <code>PrintAnswerToLife</code> باعث می‌شود اجرای <code>PrintAnswerToLife</code> متوقف شود و کنترل به <code>Go</code> بازگردد.</p></li></ul></li><li><p><strong>فراخوانی <code>GetAnswerToLife</code>:</strong></p><ul><li><p><code>GetAnswerToLife</code> فراخوانی می‌شود.</p></li><li><p><code>Task.Delay(5000)</code> فراخوانی می‌شود و یک <code>Task</code> برمی‌گرداند.</p></li><li><p><code>await task</code> در <code>GetAnswerToLife</code> باعث می‌شود اجرای <code>GetAnswerToLife</code> متوقف شود و کنترل به <code>PrintAnswerToLife</code> بازگردد.</p></li></ul></li></ol>
<h4>ب) <strong>فاز ناهمزمان (Asynchronous Phase):</strong></h4>
<ol start="1"><li><p><strong>تأخیر ۵ ثانیه‌ای (<code>Task.Delay(5000)</code>):</strong></p><ul><li><p>پس از ۵ ثانیه، <code>Task.Delay(5000)</code> کامل می‌شود.</p></li><li><p>اجرای <code>GetAnswerToLife</code> از نقطهٔ <code>await</code> ادامه می‌یابد.</p></li><li><p>مقدار <code>answer</code> محاسبه می‌شود (<code>21 * 2 = 42</code>).</p></li><li><p><code>GetAnswerToLife</code> کامل می‌شود و نتیجه (<code>42</code>) را برمی‌گرداند.</p></li></ul></li><li><p><strong>ادامهٔ اجرای <code>PrintAnswerToLife</code>:</strong></p><ul><li><p><code>PrintAnswerToLife</code> از نقطهٔ <code>await</code> ادامه می‌یابد.</p></li><li><p>مقدار <code>answer</code> (یعنی <code>42</code>) چاپ می‌شود.</p></li><li><p><code>PrintAnswerToLife</code> کامل می‌شود.</p></li></ul></li><li><p><strong>ادامهٔ اجرای <code>Go</code>:</strong></p><ul><li><p><code>Go</code> از نقطهٔ <code>await</code> ادامه می‌یابد.</p></li><li><p><code>"Done"</code> چاپ می‌شود.</p></li><li><p><code>Go</code> کامل می‌شود.</p></li></ul></li></ol>
</div>

### **Parallelism**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۱. <strong>موازی‌سازی با عدم استفاده از <code>await</code>:</strong></h3>
<p>وقتی یک متد ناهمزمان (<code>async</code>) را فراخوانی می‌کنید اما بلافاصله از <code>await</code> استفاده نمی‌کنید، کدی که بعد از آن می‌آید می‌تواند <strong>به‌صورت موازی</strong> اجرا شود. این موضوع به ویژه در برنامه‌های UI مفید است، زیرا به برنامه اجازه می‌دهد تا <strong>پاسخگو (responsive)</strong> باقی بماند.</p>
</div>

In [ ]:
_button.Click += (sender, args) => Go();

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<ul><li><p>در این مثال، <code>Go</code> یک متد <code>async</code> است، اما بلافاصله <code>await</code> نمی‌شود. این باعث می‌شود که UI همچنان پاسخگو باشد و کاربر بتواند با برنامه تعامل کند.</p></li></ul>
<h3>۲. <strong>اجرای دو عملیات ناهمزمان به‌صورت موازی:</strong></h3>
<p>برای اجرای دو عملیات ناهمزمان به‌صورت موازی، می‌توانید هر دو عملیات را شروع کنید و سپس منتظر اتمام هر دو بمانید. مثال:</p>
</div>

In [ ]:
var task1 = PrintAnswerToLife();
var task2 = PrintAnswerToLife();
await task1; 
await task2;

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4>توضیح:</h4>
<ol start="1"><li><p><strong>شروع موازی:</strong></p><ul><li><p><code>PrintAnswerToLife</code> دو بار فراخوانی می‌شود و هر دو <code>Task</code> به‌صورت موازی شروع به اجرا می‌کنند.</p></li></ul></li><li><p><strong>انتظار برای اتمام:</strong></p><ul><li><p>با استفاده از <code>await task1</code> و <code>await task2</code>، برنامه منتظر می‌ماند تا هر دو <code>Task</code> کامل شوند.</p></li></ul></li></ol>
<h4>نکته:</h4>
<p>اگر از <code>Task.WhenAll</code> استفاده کنید، می‌توانید منتظر اتمام همهٔ <code>Task</code>ها به‌صورت همزمان بمانید:</p>
</div>

In [ ]:
var task1 = PrintAnswerToLife();
var task2 = PrintAnswerToLife();

//await Task.WhenAll(task1, task2);
await Task.WaitAny(task1,task2);
//some 

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۳. <strong>تفاوت در اجرا بر اساس زمینه (Synchronization Context):</strong></h3>
<ul><li><p><strong>در UI Thread:</strong><br>اگر عملیات‌ها در یک UI Thread شروع شوند، ادامهٔ اجرا پس از <code>await</code> به UI Thread بازمی‌گردد. این باعث می‌شود که تنها <strong>پس‌زمینه‌ی عملیات</strong> (مانند <code>Task.Delay</code> یا <code>Task.Run</code>) به‌صورت موازی اجرا شوند، اما ادامهٔ کد در UI Thread اجرا می‌شود.</p></li><li><p><strong>در Thread Pool:</strong><br>اگر عملیات‌ها در یک Thread Pool شروع شوند، ادامهٔ اجرا نیز در Thread Pool انجام می‌شود و <strong>همروندی واقعی (true concurrency)</strong> اتفاق می‌افتد.</p></li></ul>

</div>

### **Asynchronous Lambda Expressions**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>عبارت‌های لامبدای ناهمزمان، لامبداهایی هستند که با کلمه کلیدی <code>async</code> علامت‌گذاری شده‌اند و می‌توانند شامل عملیات‌های ناهمزمان (<code>await</code>) باشند. این لامبداها می‌توانند به‌عنوان متدهای ناشناس استفاده شوند و یک <code>Task</code> یا <code>Task&lt;TResult&gt;</code> بازگردانند.</p>
<h3>۲. <strong>مقایسه با متدهای ناهمزمان معمولی:</strong></h3>
<h4>الف) <strong>متد ناهمزمان معمولی:</strong></h4>
</div>

In [ ]:
async Task NamedMethod()
{
    await Task.Delay(1000);
    Console.WriteLine("Foo");
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<ul><li><p>این یک متد ناهمزمان معمولی است که با <code>async</code> علامت‌گذاری شده و یک <code>Task</code> بازمی‌گرداند.</p></li></ul>
<h4>ب) <strong>عبارت لامبدای ناهمزمان:</strong></h4>
</div>

In [ ]:
Func<Task<int>> unnamed = async () =>
{
    await Task.Delay(1000);
    Console.WriteLine("Foo");
    return 40;
};

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<ul><li><p>این یک عبارت لامبدای ناهمزمان است که همان کار را انجام می‌دهد، اما به‌صورت ناشناس تعریف شده است.</p></li></ul>
<h3>۳. <strong>فراخوانی و انتظار (Await):</strong></h3>
<p>هم متدهای ناهمزمان معمولی و هم عبارت‌های لامبدای ناهمزمان را می‌توان به‌صورت مشابه فراخوانی و <code>await</code> کرد:</p>
</div>

In [ ]:
await NamedMethod(); // فراخوانی متد ناهمزمان
await unnamed();     // فراخوانی عبارت لامبدای ناهمزمان

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۴. <strong>استفاده در رویدادها (Event Handlers):</strong></h3>
<p>عبارت‌های لامبدای ناهمزمان می‌توانند به‌عنوان <strong>رویداددهنده (Event Handler)</strong> استفاده شوند. این روش بسیار مختصرتر و خوانا‌تر از تعریف یک متد جداگانه است.</p>
<h4>الف) <strong>استفاده از عبارت لامبدای ناهمزمان:</strong></h4>
</div>

In [ ]:
myButton.Click += async (sender, args) =>
{
    await Task.Delay(1000);
    myButton.Content = "Done";
};

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4>ب) <strong>استفاده از متد جداگانه:</strong></h4>
</div>

In [ ]:
myButton.Click += ButtonHandler;

async void ButtonHandler(object sender, EventArgs args)
{
    await Task.Delay(1000);
    myButton.Content = "Done";
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۵. <strong>عبارت‌های لامبدای ناهمزمان با خروجی (<code>Task&lt;TResult&gt;</code>):</strong></h3>
<p>عبارت‌های لامبدای ناهمزمان می‌توانند یک <code>Task&lt;TResult&gt;</code> بازگردانند. مثال:</p>
</div>

In [ ]:
Func<Task<int>> unnamed = async () =>
{
    await Task.Delay(1000);
    return 123;
};

int answer = await unnamed(); // answer = 123

### **Asynchronous Streams**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۱. <strong>جریان‌های ناهمزمان چیست؟</strong></h3>
<p>جریان‌های ناهمزمان ترکیبی از <strong>ایتراتورها (Iterators)</strong> و <strong>توابع ناهمزمان (Asynchronous Functions)</strong> هستند. این ویژگی به شما امکان می‌دهد تا یک دنباله‌ی ناهمزمان از داده‌ها را تولید و مصرف کنید، به‌طوری که داده‌ها به‌صورت تدریجی و در زمان در دسترس بودن تولید شوند.</p>
<h3>۲. <strong>اینترفیس‌های جریان‌های ناهمزمان:</strong></h3>
<p>برای پشتیبانی از جریان‌های ناهمزمان، دو اینترفیس اصلی معرفی شده‌اند:</p>
<p>این اینترفیس یک جریان ناهمزمان از داده‌ها را تعریف می‌کند. متد <code>GetAsyncEnumerator</code> یک <code>IAsyncEnumerator&lt;T&gt;</code> برمی‌گرداند.</p>
<h4>الف) <strong><code>IAsyncEnumerable&lt;T&gt;</code>:</strong></h4>
<p>این اینترفیس یک جریان ناهمزمان از داده‌ها را تعریف می‌کند. متد <code>GetAsyncEnumerator</code> یک <code>IAsyncEnumerator&lt;T&gt;</code> برمی‌گرداند.</p>
</div>

In [ ]:
public interface IAsyncEnumerable<out T>
{
    IAsyncEnumerator<T> GetAsyncEnumerator(...);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4>ب) <strong><code>IAsyncEnumerator&lt;T&gt;</code>:</strong></h4>
<p>این اینترفیس برای پیمایش جریان ناهمزمان استفاده می‌شود. متد <code>MoveNextAsync</code> به‌صورت ناهمزمان به عنصر بعدی می‌رود و <code>Current</code> عنصر فعلی را برمی‌گرداند.</p>
</div>

In [ ]:
public interface IAsyncEnumerator<out T> : IAsyncDisposable
{
    T Current { get; }
    ValueTask<bool> MoveNextAsync();
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4>ج) <strong><code>IAsyncDisposable</code>:</strong></h4>
<p>این اینترفیس یک نسخه‌ی ناهمزمان از <code>IDisposable</code> است و برای انجام عملیات پاک‌سازی (مانند آزادسازی منابع) استفاده می‌شود.</p>
</div>

In [ ]:
public interface IAsyncDisposable
{
    ValueTask DisposeAsync();
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۳. <strong>تولید جریان‌های ناهمزمان:</strong></h3>
<p>برای تولید یک جریان ناهمزمان، می‌توانید از ترکیب <code>yield return</code> و <code>await</code> در یک متد استفاده کنید. این متد باید <code>IAsyncEnumerable&lt;T&gt;</code> را بازگرداند.</p>
</div>

In [ ]:
async IAsyncEnumerable<int> RangeAsync(int start, int count, int delay)
{
    for (int i = start; i < start + count; i++)
    {
        await Task.Delay(delay); // تأخیر ناهمزمان
        yield return i; // تولید عنصر فعلی
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4>توضیح:</h4>
<ul><li><p>این متد یک جریان ناهمزمان از اعداد صحیح تولید می‌کند.</p></li><li><p>هر عنصر پس از یک تأخیر ناهمزمان (<code>Task.Delay</code>) تولید می‌شود.</p></li></ul>
<h3>۴. <strong>مصرف جریان‌های ناهمزمان:</strong></h3>
<p>برای مصرف جریان‌های ناهمزمان، از <code>await foreach</code> استفاده می‌کنید. این دستور به‌صورت ناهمزمان عناصر جریان را دریافت و پردازش می‌کند.</p>
</div>

In [ ]:
await foreach (var number in RangeAsync(0, 10, 500))
{
    Console.WriteLine(number);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4>توضیح:</h4>
<ul><li><p>این کد اعداد ۰ تا ۹ را با فاصله‌ی ۵۰۰ میلی‌ثانیه چاپ می‌کند.</p></li><li><p>داده‌ها به‌صورت تدریجی و در زمان در دسترس بودن تولید و مصرف می‌شوند.</p></li></ul>
<h3>۵. <strong>مقایسه با <code>Task&lt;IEnumerable&lt;T&gt;&gt;</code>:</strong></h3>
<p>در روش قدیمی‌تر (<code>Task&lt;IEnumerable&lt;T&gt;&gt;</code>)، تمام داده‌ها باید قبل از بازگشت تولید شوند. این باعث می‌شود که مصرف‌کننده‌ها تا پایان تولید داده‌ها منتظر بمانند.</p>
</div>

In [ ]:
static async Task<IEnumerable<int>> RangeTaskAsync(int start, int count, int delay)
{
    List<int> data = new List<int>();
    for (int i = start; i < start + count; i++)
    {
        await Task.Delay(delay);
        data.Add(i);
    }
    return data;
}
foreach (var data in await RangeTaskAsync(0, 10, 500))
{
    Console.WriteLine(data);
}


<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۶. <strong>مزایای جریان‌های ناهمزمان:</strong></h3>
<ol start="1"><li><p><strong>تولید تدریجی داده‌ها:</strong><br>داده‌ها به‌صورت تدریجی و در زمان در دسترس بودن تولید می‌شوند.</p></li><li><p><strong>مصرف تدریجی داده‌ها:</strong><br>مصرف‌کننده‌ها می‌توانند داده‌ها را به‌صورت تدریجی دریافت و پردازش کنند.</p></li><li><p><strong>کارایی بهتر:</strong><br>در مواردی که تولید داده‌ها زمان‌بر است، جریان‌های ناهمزمان کارایی بهتری دارند.</p></li></ol>
</div>

In [ ]:
using System;
using System.Net.Http;
using System.Threading.Tasks;

class Program
{
    static async Task Main(string[] args)
    {
        await foreach (var line in FetchHtmlLinesAsync("https://example.com/large-html-page"))
        {
            Console.WriteLine(line);
        }
    }

    static async IAsyncEnumerable<string> FetchHtmlLinesAsync(string url)
    {
        using (HttpClient client = new HttpClient())
        {
            var response = await client.GetAsync(url, HttpCompletionOption.ResponseHeadersRead);
            response.EnsureSuccessStatusCode();

            using (var stream = await response.Content.ReadAsStreamAsync())
            using (var reader = new System.IO.StreamReader(stream))
            {
                while (!reader.EndOfStream)
                {
                    var line = await reader.ReadLineAsync();
                    yield return line;
                }
            }
        }
    }
}

### **IAsyncEnumerable<T> in ASP.Net Core**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۱. <strong>جریان‌های سنتی (Traditional Streaming) در ASP.NET Core</strong></h3>
<p>ASP.NET Core به طور پیش‌فرض از جریان‌های سنتی پشتیبانی می‌کند. این جریان‌ها معمولاً برای ارسال داده‌های باینری (مانند فایل‌ها، ویدیوها، یا داده‌های بزرگ) استفاده می‌شوند. برای مثال، می‌توانید یک <code>FileStream</code> را به عنوان پاسخ بازگردانید و ASP.NET Core آن را به صورت تدریجی به کلاینت ارسال کند.</p>
</div>

In [ ]:
[HttpGet("download")]
public IActionResult DownloadFile()
{
    var stream = new FileStream("largefile.zip", FileMode.Open, FileAccess.Read);
    return File(stream, "application/octet-stream");
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این حالت، داده‌ها به صورت تدریجی از سرور به کلاینت ارسال می‌شوند، اما این روش برای داده‌های باینری و فایل‌ها مناسب است و برای داده‌های ساختاریافته (مانند لیستی از اشیا) کاربرد کمتری دارد.</p>
<h3>۲. <strong><code>IAsyncEnumerable&lt;T&gt;</code> در ASP.NET Core</strong></h3>
<p><code>IAsyncEnumerable&lt;T&gt;</code> یک مفهوم سطح بالاتر است که برای کار با <strong>داده‌های ساختاریافته به صورت ناهمزمان و تدریجی</strong> طراحی شده است. این روش به ویژه برای سناریوهایی مفید است که داده‌ها از یک منبع ناهمزمان (مانند پایگاه داده، API، یا پردازش‌های زمان‌بر) دریافت می‌شوند و می‌خواهید آن‌ها را به تدریج به کلاینت ارسال کنید.</p>
<h4>تفاوت‌های کلیدی:</h4>
<ul><li><p><strong>داده‌های ساختاریافته:</strong> <code>IAsyncEnumerable&lt;T&gt;</code> برای داده‌های ساختاریافته (مانند لیستی از اشیا) مناسب است، در حالی که جریان‌های سنتی بیشتر برای داده‌های باینری استفاده می‌شوند.</p></li><li><p><strong>ناهمزمانی:</strong> <code>IAsyncEnumerable&lt;T&gt;</code> به شما امکان می‌دهد داده‌ها را به صورت ناهمزمان تولید و ارسال کنید. این کار برای سناریوهایی که تولید داده‌ها زمان‌بر است (مانند پرس‌وجوهای پایگاه داده یا پردازش‌های پیچیده) بسیار مفید است.</p></li><li><p><strong>سادگی کدنویسی:</strong> استفاده از <code>IAsyncEnumerable&lt;T&gt;</code> کدنویسی را ساده‌تر می‌کند، زیرا نیازی به مدیریت دستی جریان‌ها (Streams) ندارید.</p></li></ul>

</div>

### **Optimizations**

##### **Completing synchronously**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>یک تابع ناهمزمان می‌تواند قبل از رسیدن به <code>await</code> به پایان برسد. این حالت زمانی اتفاق می‌افتد که نتیجه عملیات ناهمزمان بلافاصله در دسترس باشد و نیازی به انتظار نباشد. به این حالت <strong>همزمانی کامل</strong> گفته می‌شود.</p>
</div>

In [ ]:
static Dictionary<string,string> _cache = new Dictionary<string,string>();

async Task<string> GetWebPageAsync(string uri)
{
    string html;
    if (_cache.TryGetValue(uri, out html)) return html;
    return _cache[uri] = await new WebClient().DownloadStringTaskAsync(uri);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این مثال، اگر URI در کش موجود باشد، تابع بلافاصله مقدار کش‌شده را برمی‌گرداند و هیچ <code>await</code>ی اتفاق نمی‌افتد. این کار باعث می‌شود تابع به صورت همزمان کامل شود.</p>

<h4> <strong>بهینه‌سازی کامپایلر</strong></h4>
<p>کامپایلر سی‌شارپ از یک بهینه‌سازی برای توابع ناهمزمان استفاده می‌کند. اگر یک <code>Task</code> به صورت همزمان کامل شود، کامپایلر از یک <strong>Short-Circuit</strong> استفاده می‌کند تا از بازگشت به فراخواننده و سپس بازگشت به تابع جلوگیری کند. این کار با بررسی ویژگی <code>IsCompleted</code> روی <code>awaiter</code> انجام می‌شود.</p>
</div>

In [ ]:
var awaiter = GetWebPageAsync().GetAwaiter();
if (awaiter.IsCompleted)
    Console.WriteLine(awaiter.GetResult());
else
    awaiter.OnCompleted(() => Console.WriteLine(awaiter.GetResult()));

##### `ValueTask<T>`

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4> <strong>مشکل تخصیص حافظه در <code>Task</code> و <code>Task&lt;T&gt;</code></strong></h4>
<p><code>Task</code> و <code>Task&lt;T&gt;</code> انواع ارجاعی (Reference Types) هستند، بنابراین ایجاد یک نمونه جدید از آن‌ها نیاز به تخصیص حافظه در هیپ (Heap) دارد. این تخصیص حافظه می‌تواند باعث افزایش فشار روی جمع‌آوری زباله (Garbage Collection) شود، به ویژه در سناریوهایی که توابع ناهمزمان اغلب به صورت همزمان کامل می‌شوند.</p>
</div>

In [ ]:
static Dictionary<string,string> _cache = new Dictionary<string,string>();

async Task<string> GetWebPageAsync(string uri)
{
    string html;
    if (_cache.TryGetValue(uri, out html)) return html;
    return _cache[uri] = await new WebClient().DownloadStringTaskAsync(uri);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4> <strong>مشکل تخصیص حافظه در <code>Task</code> و <code>Task&lt;T&gt;</code></strong></h4>
<p><code>Task</code> و <code>Task&lt;T&gt;</code> انواع ارجاعی (Reference Types) هستند، بنابراین ایجاد یک نمونه جدید از آن‌ها نیاز به تخصیص حافظه در هیپ (Heap) دارد. این تخصیص حافظه می‌تواند باعث افزایش فشار روی جمع‌آوری زباله (Garbage Collection) شود، به ویژه در سناریوهایی که توابع ناهمزمان اغلب به صورت همزمان کامل می‌شوند.</p>

<h4> <strong>معرفی <code>ValueTask</code> و <code>ValueTask&lt;T&gt;</code></strong></h4>
<p>برای حل مشکل تخصیص حافظه در سناریوهای تکمیل همزمان، سی‌شارپ <code>ValueTask</code> و <code>ValueTask&lt;T&gt;</code> را معرفی کرده است. این انواع، ساختارهای مقداری (Value Types) هستند و بنابراین نیازی به تخصیص حافظه در هیپ ندارند.</p>

</div>

In [ ]:
async ValueTask<string> GetWebPageAsync(string uri)
{
    if (_cache.TryGetValue(uri, out var html)) 
        return html; // بدون ایجاد Task

    return await new WebClient().DownloadStringTaskAsync(uri);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4><strong>رفتار <code>ValueTask&lt;T&gt;</code> در حالت‌های مختلف</strong></h4>
<ul><li><p><strong>تکمیل همزمان:</strong> اگر عملیات به صورت همزمان کامل شود، <code>ValueTask&lt;T&gt;</code> هیچ تخصیص حافظه‌ای ایجاد نمی‌کند.</p></li><li><p><strong>تکمیل ناهمزمان:</strong> اگر عملیات به صورت ناهمزمان کامل شود، <code>ValueTask&lt;T&gt;</code> به طور خودکار یک <code>Task&lt;T&gt;</code> ایجاد می‌کند و از آن برای مدیریت عملیات ناهمزمان استفاده می‌کند. در این حالت، مزیت <code>ValueTask&lt;T&gt;</code> از بین می‌رود.</p></li></ul>

<h4> <strong>تبدیل <code>ValueTask&lt;T&gt;</code> به <code>Task&lt;T&gt;</code></strong></h4>
<p>می‌توانید یک <code>ValueTask&lt;T&gt;</code> را به یک <code>Task&lt;T&gt;</code> معمولی تبدیل کنید. این کار با استفاده از متد <code>AsTask</code> انجام می‌شود.</p>
</div>

In [ ]:
ValueTask<string> valueTask = GetWebPageAsync("https://example.com");

Task<string> task = valueTask.AsTask();

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3><strong>احتیاط‌های استفاده از <code>ValueTask&lt;T&gt;</code> در C#</strong></h3>
<p><code>ValueTask&lt;T&gt;</code> یک <strong>ساختار (struct)</strong> است که به دلایل <strong>عملکردی (performance)</strong> معرفی شده است، اما همین ویژگی باعث ایجاد چالش‌هایی در استفاده از آن می‌شود. برخلاف <code>Task&lt;T&gt;</code>، که یک <strong>نوع ارجاعی (reference type)</strong> است، <code>ValueTask&lt;T&gt;</code> یک <strong>نوع مقداری (value type)</strong> است، که می‌تواند رفتارهای غیرمنتظره‌ای ایجاد کند.</p>
<h3><strong>مشکلاتی که باید از آن‌ها اجتناب کرد</strong></h3>
<h4> <strong>اجتناب از <code>await</code> کردن یک <code>ValueTask&lt;T&gt;</code> بیش از یک بار</strong></h4>
<h3>۱️⃣ <strong>اجتناب از <code>await</code> کردن یک <code>ValueTask&lt;T&gt;</code> بیش از یک بار</strong></h3>
</div>

In [ ]:
ValueTask<int> task = GetValueAsync();
int result1 = await task; // ✅ اولین بار صحیح است
int result2 = await task; // ❌ خطا: مقدار ممکن است دیگر معتبر نباشد

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>🔹 در کد بالا، <code>ValueTask&lt;int&gt;</code> یک مقدار را نگه می‌دارد، اما پس از یک‌بار <code>await</code> کردن، دیگر تضمینی نیست که مقدار <strong>مجدد خوانده شود</strong>.<br>🔹 در مقایسه، <code>Task&lt;int&gt;</code> را می‌توان چندین بار <code>await</code> کرد، زیرا همواره یک شیء ارجاعی ثابت باقی می‌ماند.</p>
<h4><strong>راه‌حل:</strong></h4>
<p>اگر به <code>ValueTask&lt;T&gt;</code> نیاز دارید که بیش از یک‌بار استفاده شود، ابتدا آن را به <code>Task&lt;T&gt;</code> تبدیل کنید:</p>
</div>

In [ ]:
ValueTask<int> valueTask = GetValueAsync();
Task<int> task = valueTask.AsTask();
int result1 = await task;
int result2 = await task; // اکنون مشکلی ندارد

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h4><strong>اجتناب از <code>GetAwaiter().GetResult()</code> زمانی که عملیات هنوز کامل نشده است</strong></h4>
<p>🔹 <code>GetAwaiter().GetResult()</code> برای اجرای <strong>همگام (synchronous)</strong> یک <code>Task</code> یا <code>ValueTask</code> استفاده می‌شود.<br>🔹 اگر <code>ValueTask&lt;T&gt;</code> هنوز کامل نشده باشد و <code>GetResult()</code> روی آن اجرا شود، ممکن است <strong>رفتار نامشخصی</strong> ایجاد کند.</p>
</div>

In [ ]:
ValueTask<int> task = GetValueAsync();
int result = task.GetAwaiter().GetResult(); // ❌ اگر هنوز مقدار آماده نباشد، خطای ناهمگام رخ می‌دهد!

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>🔹 <code>ValueTask&lt;T&gt;</code> ممکن است یا <strong>یک مقدار فوری داشته باشد</strong> یا <strong>هنوز در حال پردازش باشد</strong>. اگر هنوز آماده نباشد و <code>GetResult()</code> را فراخوانی کنیم، احتمالاً <strong>یک بلاکینگ ناخواسته یا حتی کرش برنامه</strong> رخ می‌دهد.</p>
<h4><strong>راه‌حل:</strong></h4>
<p>اگر باید از <code>GetResult()</code> استفاده کنید، ابتدا <code>ValueTask&lt;T&gt;</code> را به <code>Task&lt;T&gt;</code> تبدیل کنید:</p>
</div>

In [ ]:
int result = GetValueAsync().AsTask().GetAwaiter().GetResult();

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3><strong>چرا این مشکلات وجود دارند؟</strong></h3>
<p>🔹 <code>ValueTask&lt;T&gt;</code> برخلاف <code>Task&lt;T&gt;</code> <strong>یک مقدار مقداری (struct)</strong> است و از روی آن <strong>کپی‌های جدیدی</strong> ساخته می‌شود.<br>🔹 این تفاوت باعث می‌شود که <strong>رفتار آن در مقایسه با <code>Task&lt;T&gt;</code> متفاوت باشد و برخی از الگوهای رایج در <code>Task&lt;T&gt;</code> برای <code>ValueTask&lt;T&gt;</code> خطرناک شوند.</strong></p>

<h3><strong>جمع‌بندی و توصیه‌ها</strong></h3>
<p>✅ <strong>چه زمانی از <code>ValueTask&lt;T&gt;</code> استفاده کنیم؟</strong></p>
<ul><li>زمانی که تابع اغلب <strong>مقدار فوری برمی‌گرداند</strong> (مثلاً مقدار کش شده) و فقط گاهی یک <code>Task</code> برمی‌گرداند.</li><li>در سناریوهایی که هزینه‌ی ایجاد شیء <code>Task&lt;T&gt;</code> بالا است.</li></ul>
<p>❌ <strong>چه زمانی نباید از <code>ValueTask&lt;T&gt;</code> استفاده کنیم؟</strong></p>
<ul><li>وقتی مقدار بازگشتی باید چند بار <code>await</code> شود.</li><li>وقتی می‌خواهیم <code>GetAwaiter().GetResult()</code> را اجرا کنیم.</li><li>در سناریوهایی که مدیریت <code>Task&lt;T&gt;</code> ساده‌تر است.</li></ul>
</div>